In [1]:
#!/usr/bin/env julia
"""
inject_bank_links.jl

Scans all lecture notebooks (L##.ipynb) and inserts a practice link

    [Practice](../exercises/random-2.html#bank-exr-label){.bank-link}

into every ::: {#exr-...} block that doesn't already have one.
Modifies notebooks in-place. Safe to re-run — skips exercises that
already have the link.

Usage:
    julia inject_bank_links.jl
"""

using JSON

# ── Configuration ──────────────────────────────────────────────────────────────
lectures_dir  = "../lectures-test"
bank_link_url = "../exercises/bank.html"

# ── Helpers ────────────────────────────────────────────────────────────────────

"""
    inject_links_into_source(lines) -> (new_lines, n_injected)

Given the source lines of a markdown cell, find every ::: {#exr-...}
block and insert a bank link on the first blank line after the opening
fence if one isn't already present.
"""
function inject_links_into_source(lines::Vector{<:AbstractString})
    out        = String[]
    n          = length(lines)
    i          = 1
    n_injected = 0

    while i <= n
        line = lines[i]
        m    = match(r"^:::\s*\{#(exr-[\w-]+)", line)

        if m !== nothing
            exr_id   = m.captures[1]
            bank_url = "$bank_link_url#bank-$exr_id"
            link_str = "[]($bank_url){.bank-link}"

            push!(out, line)  # the opening ::: line
            i += 1

            # Collect the block content, tracking depth
            block_lines = String[]
            depth = 1
            while i <= n && depth > 0
                l = lines[i]
                if occursin(r"^:::\s*\{", l)
                    depth += 1
                elseif strip(l) == ":::"
                    depth -= 1
                end
                depth > 0 && push!(block_lines, l)
                i += 1
            end

            # Check if link already exists anywhere in the block
            already_has_link = any(occursin("bank-$exr_id", bl) for bl in block_lines)

            if already_has_link
                # Write block unchanged
                append!(out, block_lines)
                push!(out, ":::")
            else
                # Insert link after the first non-empty line (the title/label line),
                # preceded and followed by a blank line
                insert_at = 1
                for (j, bl) in enumerate(block_lines)
                    if !isempty(strip(bl))
                        insert_at = j + 1
                        break
                    end
                end

                new_block = copy(block_lines)
                # Ensure there's a blank line before and after the link
                insert!(new_block, insert_at, "")
                insert!(new_block, insert_at, link_str)
                insert!(new_block, insert_at, "")

                append!(out, new_block)
                push!(out, ":::")
                n_injected += 1
            end
        else
            push!(out, line)
            i += 1
        end
    end

    return out, n_injected
end

"""
    process_notebook(path) -> n_injected

Load a notebook, inject bank links into all markdown cells, write back
in-place if anything changed. Returns the number of links injected.
"""
function process_notebook(path::String)::Int
    nb          = JSON.parsefile(path)
    total       = 0
    any_changed = false

    for cell in nb["cells"]
        cell["cell_type"] == "markdown" || continue

        # cell["source"] is a Vector of strings (each ending in \n except the last)
        # Split into lines for processing, then rejoin
        raw   = join(cell["source"])
        lines = split(raw, '\n')

        new_lines, n = inject_links_into_source(lines)

        if n > 0
            # Reconstruct source as a JSON array of strings with \n terminators
            rejoined = join(new_lines, '\n')
            # Quarto/Jupyter stores source as array of lines ending with \n
            src_lines = split(rejoined, '\n')
            cell["source"] = [
                i < length(src_lines) ? src_lines[i] * "\n" : src_lines[i]
                for i in 1:length(src_lines)
            ]
            total      += n
            any_changed = true
        end
    end

    if any_changed
        open(path, "w") do io
            JSON.print(io, nb, 2)
        end
    end

    return total
end

# ── Main ───────────────────────────────────────────────────────────────────────

function main()
    pattern   = r"^L\d+\.ipynb$"
    all_files = filter(f -> occursin(pattern, f), readdir(lectures_dir))
    sort!(all_files)
    all_files = filter(f -> f != "L00.ipynb", all_files)

    if isempty(all_files)
        @warn "No lecture notebooks found in $lectures_dir"
        return
    end

    println("Scanning $(length(all_files)) notebook(s)...\n")
    total = 0
    for fname in all_files
        path = joinpath(lectures_dir, fname)
        n    = process_notebook(path)
        if n > 0
            println("  $fname: injected $n link(s)")
        else
            println("  $fname: nothing to do")
        end
        total += n
    end
    println("\nDone. $total link(s) injected total.")
end

main()

Scanning 1 notebook(s)...

  L01.ipynb: injected 6 link(s)

Done. 6 link(s) injected total.
